<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Bicicletas Públicas, de la Estrella al Lago 🚲
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 02
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/02_Data_Warehouse_Data_Lake_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 02: *Data Warehousing* y *Data Lake*** ([README del módulo](../02%20-%20Data%20Warehousing%20y%20Data%20Lake/README.md)) con un problema **nuevo**: los viajes de una **red ficticia de bicicletas públicas** («BiciAndina») que se analizan primero en un **almacén dimensional** y luego se guardan en un **lago** de archivos Parquet.

Al terminar serás capaz de:

1. Diseñar un **ETL con reglas de calidad**: corregir lo inequívoco, mandar a **cuarentena con su motivo** lo demás y verificar la **conservación de filas**.
2. Modelar un **esquema estrella** en DuckDB (claves sustitutas, dimensión *role-playing*, miembro «No identificado», `PRIMARY KEY`, `FOREIGN KEY` y `CHECK`) y comprobar que **no hay huérfanos**.
3. Hacer **operaciones OLAP** (*roll-up*, *slice*, *dice* y *pivot*) en SQL y en `pandas`, y conciliarlas entre sí.
4. Escribir un **lago Parquet particionado estilo Hive**, aplicar **poda de particiones**, **compactar** archivos pequeños y leer con **esquema en lectura** una columna que aparece tarde.

> 📚 **Repasa si lo necesitas:** [DW, OLAP, ETL y esquema estrella](../02%20-%20Data%20Warehousing%20y%20Data%20Lake/00_Data_Warehousing_OLAP_ETL_y_Esquema_Estrella.ipynb) · [Data Lake, Hive y Pig](../02%20-%20Data%20Warehousing%20y%20Data%20Lake/01_Data_Lake_Hive_y_Pig.ipynb) · [Comparación DW frente a Data Lake](../02%20-%20Data%20Warehousing%20y%20Data%20Lake/02_Comparacion_Data_Warehouse_vs_Data_Lake.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; la red y sus viajes son **inventados**). Bibliotecas: `pandas`, `numpy`, `duckdb` y `pyarrow` (el motor de Parquet de `pandas`). El lago se escribe en un **directorio temporal** que se borra al cerrar el núcleo. Los retos 3 y 4 usan lo que construyas en el reto 2 y en los datos de ejemplo `viajes`, no dependen del reto 1.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import atexit
import os
import shutil
import tempfile
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

SEMILLA = 33

estaciones = pd.DataFrame({
    "codigo": [f"EST-{i:02d}" for i in range(1, 11)],
    "nombre": ["Plaza Mayor", "Terminal", "Parque Norte", "Hospital", "Universidad",
               "Mercado", "Estadio", "Biblioteca", "Parque Sur", "Estación Férrea"],
    "barrio": ["Centro", "Occidente", "Norte", "Centro", "Norte", "Sur", "Occidente", "Centro", "Sur", "Sur"],
})


def generar_limpios(semilla, n=3000, f0="2025-01-01", dias=181):
    """Viajes ya limpios (sin defectos), repartidos en `dias` días desde `f0`."""
    rng = np.random.default_rng(semilla)
    inicio = (pd.Timestamp(f0) + pd.to_timedelta(rng.integers(0, dias * 86400, n), unit="s")).floor("min")
    o = rng.integers(0, 10, n)
    d = (o + rng.integers(1, 10, n)) % 10
    dur = rng.integers(3, 90, n)
    return pd.DataFrame({
        "id_viaje": [f"V{semilla % 100:02d}{k:05d}" for k in range(1, n + 1)],
        "inicio": inicio,
        "est_origen": estaciones["codigo"].to_numpy()[o],
        "est_destino": estaciones["codigo"].to_numpy()[d],
        "plan": rng.choice(np.array(["casual", "anual", "estudiante", None], dtype=object), n, p=[0.45, 0.25, 0.20, 0.10]),
        "duracion_min": dur,
        "tarifa_cop": 1000.0 + 100.0 * (dur // 5),
    })


def generar_crudos(semilla=SEMILLA):
    """Exportación del POS con defectos sembrados: 12 ids repetidos, 9 estaciones desconocidas,
    7 duraciones no positivas, 6 tarifas vacías, 5 fechas ilegibles (a cuarentena) y
    40 códigos mal escritos, 60 fechas dd/mm/aaaa y 30 planes con otra grafía (corregibles)."""
    rng = np.random.default_rng(semilla + 100)
    b = generar_limpios(semilla + 1)
    cr = b.assign(inicio=b["inicio"].dt.strftime("%Y-%m-%d %H:%M"))
    idx = rng.permutation(len(cr))
    cr.loc[idx[:9], "est_destino"] = "EST-99"
    cr.loc[idx[9:16], "duracion_min"] = rng.choice([0, -5], 7)
    cr.loc[idx[16:22], "tarifa_cop"] = np.nan
    cr.loc[idx[22:27], "inicio"] = "ayer en la tarde"
    f_cod = idx[27:67]
    cr.loc[f_cod, "est_origen"] = [(" " if k % 2 else "") + cr.at[i, "est_origen"].lower() + (" " if k % 2 else "") for k, i in enumerate(f_cod)]
    f_fec = idx[67:127]
    cr.loc[f_fec, "inicio"] = pd.to_datetime(cr.loc[f_fec, "inicio"]).dt.strftime("%d/%m/%Y %H:%M")
    resto = idx[127:]
    f_plan = resto[cr.loc[resto, "plan"].notna().to_numpy()][:30]
    cr.loc[f_plan, "plan"] = [(" " + p.upper() + " ") if k % 2 else p.capitalize() for k, p in enumerate(cr.loc[f_plan, "plan"])]
    d_src = resto[~np.isin(resto, f_plan)][:12]
    return pd.concat([cr, cr.loc[d_src]], ignore_index=True)


crudos = generar_crudos()                       # reto 1: exportación sucia (3 012 filas)
viajes = generar_limpios(SEMILLA + 2)            # retos 2-4: viajes ya limpios (3 000 filas, ene-jun 2025)
lote_nuevo = generar_limpios(SEMILLA + 3, n=300, f0="2025-07-01", dias=31).assign(bateria_pct=lambda d: (d.index % 90 + 10))
con = duckdb.connect()                           # almacén en memoria (retos 2 y 3)
RAIZ_LAGO = tempfile.mkdtemp(prefix="lago_taller_")   # lago de archivos (reto 4), se borra al cerrar el núcleo
atexit.register(shutil.rmtree, RAIZ_LAGO, ignore_errors=True)

print("Exportación cruda:", crudos.shape, "| viajes limpios:", viajes.shape, "| lote de julio:", lote_nuevo.shape)
print("Lago temporal en:", RAIZ_LAGO)

---
## Reto 1: Un ETL que corrige, rechaza y cuadra 🧹

**Contexto:** la exportación del sistema de pagos (`crudos`) llega con defectos. Unos se pueden **corregir sin ambigüedad**; otros no, y esas filas van a **cuarentena con su motivo** (no se borran).

**Enunciado:** implementa `transformar(crudos)` → `(limpios, cuarentena, correcciones)`.
1. **Corregir:** `est_origen` y `est_destino` con `strip()` y mayúsculas; `plan` con `strip()` y minúsculas (los nulos se conservan: son usuarios sin identificar); `inicio` en formato `aaaa-mm-dd HH:MM` **o** `dd/mm/aaaa HH:MM`, convertido a `datetime`.
2. **Rechazar** a `cuarentena` (con una columna `motivo`, conservando los valores originales), con esta prioridad: `"duplicado"` (mismo `id_viaje`, se conserva la primera aparición), `"fecha_ilegible"` (ninguno de los dos formatos), `"estacion_desconocida"` (código fuera de `estaciones`), `"duracion_invalida"` (`duracion_min <= 0`) y `"tarifa_nula"`.
3. `correcciones`: diccionario con cuántas celdas se corrigieron: `"codigo_estacion"`, `"formato_fecha"` (fechas `dd/mm/aaaa`) y `"plan_caso"`.
4. Guarda el resultado en `limpios`, `cuarentena` y `correcciones`, y arma `recibo = {"entran": ..., "limpios": ..., "cuarentena": ...}`.

**Criterios de aceptación:** `entran = limpios + cuarentena` (conservación); cada regla encuentra **exactamente** lo sembrado; los corregidos quedan en `limpios`; y `limpios` es único por `id_viaje`, con `inicio` de tipo fecha y hora.

**Puntaje sugerido:** 20 puntos.

In [ ]:
def transformar(crudos):
    # TODO: corrige lo inequívoco, manda el resto a cuarentena con `motivo` y cuenta las correcciones
    # devuelve (limpios, cuarentena, correcciones)
    return None


# TODO: completa
limpios = cuarentena = correcciones = None
recibo = None      # {"entran": ..., "limpios": ..., "cuarentena": ...}

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert limpios is not None and cuarentena is not None and isinstance(correcciones, dict), "transformar debe devolver (limpios, cuarentena, correcciones) y debes guardarlos en esas variables."
assert "motivo" in cuarentena.columns, "La cuarentena debe llevar la columna 'motivo'."
_cod = lambda s: s.str.strip().str.upper()
_dup = crudos.duplicated("id_viaje")
_iso = pd.to_datetime(crudos["inicio"], format="%Y-%m-%d %H:%M", errors="coerce")
_lat = pd.to_datetime(crudos["inicio"], format="%d/%m/%Y %H:%M", errors="coerce")
_ileg = ~_dup & _iso.isna() & _lat.isna()
_desc = ~_dup & ~_ileg & ~(_cod(crudos["est_origen"]).isin(estaciones["codigo"]) & _cod(crudos["est_destino"]).isin(estaciones["codigo"]))
_dur = ~_dup & ~_ileg & ~_desc & (crudos["duracion_min"] <= 0)
_tar = ~_dup & ~_ileg & ~_desc & ~_dur & crudos["tarifa_cop"].isna()
_esp = {"duplicado": int(_dup.sum()), "fecha_ilegible": int(_ileg.sum()), "estacion_desconocida": int(_desc.sum()), "duracion_invalida": int(_dur.sum()), "tarifa_nula": int(_tar.sum())}
assert _esp == {"duplicado": 12, "fecha_ilegible": 5, "estacion_desconocida": 9, "duracion_invalida": 7, "tarifa_nula": 6}, "Los defectos sembrados cambiaron: revisa la celda de preparación."
assert cuarentena["motivo"].value_counts().to_dict() == _esp, f"Cada regla debe encontrar exactamente lo sembrado, con la prioridad pedida; esperaba {_esp}."
assert recibo == {"entran": len(crudos), "limpios": len(limpios), "cuarentena": len(cuarentena)} and recibo["entran"] == recibo["limpios"] + recibo["cuarentena"] == 3012, "El recibo debe cumplir entran = limpios + cuarentena (3 012 = 2 973 + 39)."
_ok = ~(_dup | _ileg | _desc | _dur | _tar)
assert set(limpios["id_viaje"]) == set(crudos.loc[_ok, "id_viaje"]) and limpios["id_viaje"].is_unique, "limpios debe contener exactamente las filas que ninguna regla rechaza, una por id_viaje."
assert pd.api.types.is_datetime64_any_dtype(limpios["inicio"]), "limpios['inicio'] debe ser de tipo fecha y hora (pd.to_datetime)."
_esp_f = _iso.fillna(_lat)[_ok].to_numpy()
assert (limpios.set_index("id_viaje").loc[crudos.loc[_ok, "id_viaje"], "inicio"].to_numpy() == _esp_f).all(), "Las fechas dd/mm/aaaa deben interpretarse con día primero y las aaaa-mm-dd tal cual."
assert set(limpios["est_origen"]) | set(limpios["est_destino"]) <= set(estaciones["codigo"]), "Los códigos de estación corregidos deben quedar como 'EST-NN' (strip + mayúsculas)."
assert set(limpios["plan"].dropna()) == {"casual", "anual", "estudiante"} and limpios["plan"].isna().sum() == crudos.loc[_ok, "plan"].isna().sum(), "plan: strip + minúsculas; los nulos (usuarios sin identificar) se conservan."
assert (limpios["duracion_min"] > 0).all() and limpios["tarifa_cop"].notna().all(), "limpios no puede tener duraciones no positivas ni tarifas vacías."
_cf = int((crudos["est_origen"] != _cod(crudos["est_origen"])).sum() + (crudos["est_destino"] != _cod(crudos["est_destino"])).sum())
_pl = int((crudos["plan"].notna() & (crudos["plan"] != crudos["plan"].str.strip().str.lower())).sum())
assert correcciones == {"codigo_estacion": _cf, "formato_fecha": int((_iso.isna() & _lat.notna()).sum()), "plan_caso": _pl} == {"codigo_estacion": 40, "formato_fecha": 60, "plan_caso": 30}, "correcciones debe contar celdas corregidas: 40 códigos, 60 fechas dd/mm/aaaa y 30 planes."
print(f"✅ Reto 1 superado: recibo {recibo} | motivos {_esp} | correcciones {correcciones}")

---
## Reto 2: El esquema estrella de los viajes ⭐

**Contexto:** con los `viajes` limpios de ejemplo vas a construir el almacén en la conexión `con` (DuckDB). La **granularidad** del hecho es *un viaje*.

**Enunciado**
1. Crea las tablas (con `PRIMARY KEY`, `NOT NULL`, `FOREIGN KEY` y `CHECK`):
   - `dim_estacion(sk_estacion, codigo, nombre, barrio)`: `sk_estacion` = 1..10 en el orden de `estaciones`; `codigo` único.
   - `dim_fecha(sk_fecha, fecha, anio, mes, dia_semana, es_fin_de_semana)`: una fila por día del 2025-01-01 al 2025-06-30; `sk_fecha` = entero `aaaammdd`; `dia_semana` de 1 (lunes) a 7 (domingo); fin de semana = sábado o domingo.
   - `dim_usuario(sk_usuario, plan)`: `sk_usuario = 0` es el miembro **«No identificado»** (planes nulos); los planes distintos, ordenados alfabéticamente, van de 1 en adelante.
   - `fact_viaje(id_viaje, sk_fecha, sk_estacion_origen, sk_estacion_destino, sk_usuario, hora, duracion_min, tarifa_cop)`: `id_viaje` es la **dimensión degenerada** (clave primaria); las cuatro claves sustitutas son `FOREIGN KEY` (¡origen y destino apuntan a la **misma** dimensión!); `CHECK (duracion_min > 0)` y `CHECK (tarifa_cop >= 0)`.
2. Carga las dimensiones desde `pandas` (`con.register`) y el hecho con un `INSERT ... SELECT` que **resuelve las claves sustitutas** con `JOIN` (y `LEFT JOIN` + `COALESCE` para el usuario).
3. `n_fact`: filas del hecho. `huerfanos`: total de filas del hecho sin su fila de dimensión (suma de los cuatro casos, con `LEFT JOIN ... IS NULL`). `total_sql`: suma de `tarifa_cop` en el hecho.

**Criterios de aceptación:** las tablas existen con las columnas pedidas; las restricciones **funcionan** (la base rechaza un id repetido, una clave inexistente, una duración 0 y una tarifa negativa); no hay huérfanos y el hecho suma lo mismo que `viajes`.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: crea las cuatro tablas (con.execute("""CREATE TABLE ...""")). Primero las dimensiones, luego el hecho.


# TODO: carga las dimensiones (construye un DataFrame por dimensión, regístralo con con.register y usa INSERT ... SELECT)


# TODO: carga el hecho resolviendo las claves sustitutas con JOIN


# TODO: completa
n_fact = huerfanos = total_sql = None

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_tablas = {r[0] for r in con.execute("SELECT table_name FROM information_schema.tables").fetchall()}
assert {"dim_estacion", "dim_fecha", "dim_usuario", "fact_viaje"} <= _tablas, "Faltan tablas: crea dim_estacion, dim_fecha, dim_usuario y fact_viaje con con.execute(...)."
_req = {"dim_estacion": {"sk_estacion", "codigo", "nombre", "barrio"}, "dim_fecha": {"sk_fecha", "fecha", "anio", "mes", "dia_semana", "es_fin_de_semana"},
        "dim_usuario": {"sk_usuario", "plan"}, "fact_viaje": {"id_viaje", "sk_fecha", "sk_estacion_origen", "sk_estacion_destino", "sk_usuario", "hora", "duracion_min", "tarifa_cop"}}
for _t, _cols in _req.items():
    _have = {r[0] for r in con.execute(f"SELECT column_name FROM information_schema.columns WHERE table_name = '{_t}'").fetchall()}
    assert _cols <= _have, f"{_t} debe tener las columnas {sorted(_cols)}; le faltan {sorted(_cols - _have)}."
_cons = con.execute("SELECT constraint_type, table_name FROM duckdb_constraints()").fetchall()
assert sum(1 for t, n in _cons if t == "FOREIGN KEY" and n == "fact_viaje") >= 4, "fact_viaje debe declarar 4 FOREIGN KEY (fecha, origen, destino y usuario)."
assert ("PRIMARY KEY", "fact_viaje") in _cons and ("CHECK", "fact_viaje") in _cons, "fact_viaje debe tener PRIMARY KEY (id_viaje) y restricciones CHECK."
assert con.execute("SELECT count(*) FROM dim_estacion").fetchone()[0] == 10 and con.execute("SELECT min(sk_estacion), max(sk_estacion) FROM dim_estacion").fetchone() == (1, 10), "dim_estacion: 10 filas con sk_estacion de 1 a 10."
assert con.execute("SELECT count(*) FROM dim_fecha").fetchone()[0] == 181, "dim_fecha: una fila por día del 2025-01-01 al 2025-06-30 (181 días)."
assert con.execute("SELECT dia_semana, es_fin_de_semana FROM dim_fecha WHERE sk_fecha IN (20250106, 20250111, 20250112) ORDER BY sk_fecha").fetchall() == [(1, False), (6, True), (7, True)], "dia_semana: 1 = lunes ... 7 = domingo (6 y 7 son fin de semana). El 2025-01-06 fue lunes."
assert con.execute("SELECT sk_usuario, plan FROM dim_usuario ORDER BY sk_usuario").fetchall() == [(0, "No identificado"), (1, "anual"), (2, "casual"), (3, "estudiante")], "dim_usuario: 0 = 'No identificado' y luego los planes en orden alfabético."
assert n_fact == len(viajes) == 3000 and huerfanos == 0, "El hecho debe tener una fila por viaje (3 000) y ningún huérfano."
assert abs(total_sql - viajes["tarifa_cop"].sum()) < 1e-6, "El hecho debe sumar lo mismo que 'viajes' (conciliación)."
assert con.execute("SELECT count(*) FROM fact_viaje WHERE sk_usuario = 0").fetchone()[0] == int(viajes["plan"].isna().sum()), "Los viajes sin plan deben apuntar al miembro 0 «No identificado» (no se pierden)."
con.register("_chk_viajes", viajes)
assert con.execute("SELECT count(*) FROM fact_viaje f JOIN _chk_viajes v ON v.id_viaje = f.id_viaje WHERE f.hora <> hour(v.inicio) OR f.duracion_min <> v.duracion_min").fetchone()[0] == 0, "hora debe ser la hora de inicio del viaje (0-23) y duracion_min la del viaje original."
assert con.execute("SELECT count(*) FROM fact_viaje f JOIN dim_estacion e ON e.sk_estacion = f.sk_estacion_origen WHERE e.codigo <> (SELECT est_origen FROM _chk_viajes v WHERE v.id_viaje = f.id_viaje)").fetchone()[0] == 0, "La estación de origen del hecho no coincide con la de 'viajes': revisa el JOIN de origen (y que origen y destino no estén cruzados)."


def _rechaza(col, valor):
    sel = ", ".join(f"{valor} AS {c}" if c == col else c for c in ["id_viaje", "sk_fecha", "sk_estacion_origen", "sk_estacion_destino", "sk_usuario", "hora", "duracion_min", "tarifa_cop"])
    con.execute("BEGIN")
    try:
        con.execute(f"INSERT INTO fact_viaje (id_viaje, sk_fecha, sk_estacion_origen, sk_estacion_destino, sk_usuario, hora, duracion_min, tarifa_cop) SELECT {sel} FROM fact_viaje LIMIT 1")
        return False
    except duckdb.Error:
        return True
    finally:
        con.execute("ROLLBACK")


_id = con.execute("SELECT id_viaje FROM fact_viaje LIMIT 1").fetchone()[0]
assert _rechaza("id_viaje", f"'{_id}'"), "La base debe rechazar un id_viaje repetido (PRIMARY KEY)."
assert _rechaza("sk_fecha", "29991231") and _rechaza("sk_usuario", "99") and _rechaza("sk_estacion_destino", "99"), "La base debe rechazar claves sustitutas inexistentes (FOREIGN KEY en fecha, usuario y destino)."
assert _rechaza("duracion_min", "0") and _rechaza("tarifa_cop", "-1.0"), "La base debe rechazar duracion_min = 0 y tarifa_cop < 0 (CHECK)."
assert not _rechaza("id_viaje", "'ZZ-NUEVO'"), "Una fila válida con un id nuevo sí debe aceptarse: tus restricciones son demasiado estrictas."
assert con.execute("SELECT count(*) FROM fact_viaje").fetchone()[0] == 3000, "Las pruebas de restricciones no deben dejar filas nuevas en el hecho."
print(f"✅ Reto 2 superado: {n_fact:,} viajes en el hecho, 0 huérfanos, total {total_sql:,.0f} COP | restricciones PK, FK y CHECK funcionan")

---
## Reto 3: Preguntas OLAP en SQL y en `pandas` 🧊

**Contexto:** con la estrella del reto 2, la gerencia quiere mirar los ingresos desde varios ángulos. Resolverás cada vista en **SQL** y verificarás que **`pandas` da lo mismo**.

**Enunciado** (cada consulta es una cadena SQL; ejecútala con `con.execute(sql).df()`)
1. `sql_rollup` → `df_rollup`: viajes e ingreso por **mes** y **barrio de origen** con `GROUP BY ROLLUP (mes, barrio)`; columnas `mes`, `barrio`, `viajes`, `ingreso`. Las filas con `NULL` son subtotales y el total general.
2. `sql_slice` → `df_slice`: **rebanada** de `mes = 3` (marzo): `dia_semana` y `viajes`, ordenada por `dia_semana`.
3. `sql_dice` → `df_dice`: **dado** con `plan IN ('anual', 'estudiante')` y `mes IN (3, 4)`: columnas `plan`, `mes`, `viajes`, `ingreso`, agrupado por `plan` y `mes`.
4. `sql_cubo` → `cubo`: una fila por viaje con `mes`, `barrio_origen`, `plan` y `tarifa_cop`. Con `pandas`: `pivote = cubo.pivot_table(...)` con `barrio_origen` en filas, `plan` en columnas, la **suma** de `tarifa_cop`, totales (`margins=True`, `margins_name="Total"`) y ceros donde no haya datos.

**Criterios de aceptación:** el *roll-up* tiene una fila de detalle por (mes, barrio), una de subtotal por mes y **un total general**, y todas coinciden con `pandas.groupby`; la rebanada y el dado coinciden con `pandas`; y el pivote, con sus márgenes, cuadra con el total del hecho.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa
sql_rollup = sql_slice = sql_dice = sql_cubo = None   # cadenas SQL sobre la estrella

df_rollup = df_slice = df_dice = cubo = None          # con.execute(sql).df()
pivote = None                                         # cubo.pivot_table(...)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
for _n, _d in [("df_rollup", df_rollup), ("df_slice", df_slice), ("df_dice", df_dice), ("cubo", cubo)]:
    assert isinstance(_d, pd.DataFrame), f"{_n} debe ser el DataFrame que devuelve con.execute(sql).df()."
_v = viajes.assign(mes=viajes["inicio"].dt.month, dia_semana=viajes["inicio"].dt.dayofweek + 1, plan=viajes["plan"].fillna("No identificado")).merge(
    estaciones.rename(columns={"codigo": "est_origen", "barrio": "barrio_origen"})[["est_origen", "barrio_origen"]], on="est_origen")
assert list(df_rollup.columns) == ["mes", "barrio", "viajes", "ingreso"], "df_rollup debe tener exactamente las columnas mes, barrio, viajes e ingreso."
_det = _v.groupby(["mes", "barrio_origen"])["tarifa_cop"].agg(["count", "sum"])
_sub = _v.groupby("mes")["tarifa_cop"].agg(["count", "sum"])
_r = df_rollup
_det_r = _r[_r["mes"].notna() & _r["barrio"].notna()].set_index(["mes", "barrio"])
_sub_r = _r[_r["mes"].notna() & _r["barrio"].isna()].set_index("mes")
_tot_r = _r[_r["mes"].isna() & _r["barrio"].isna()]
assert len(_det_r) == len(_det) and (_det_r["viajes"].to_numpy() == _det["count"].to_numpy()).all() and np.allclose(_det_r["ingreso"].to_numpy(), _det["sum"].to_numpy()), "Las filas de detalle del roll-up deben coincidir con groupby(['mes', 'barrio_origen'])."
assert len(_sub_r) == len(_sub) and (_sub_r["viajes"].to_numpy() == _sub["count"].to_numpy()).all() and np.allclose(_sub_r["ingreso"].to_numpy(), _sub["sum"].to_numpy()), "Los subtotales por mes (barrio = NULL) deben coincidir con groupby('mes')."
assert len(_tot_r) == 1 and int(_tot_r["viajes"].iloc[0]) == len(viajes) and abs(_tot_r["ingreso"].iloc[0] - viajes["tarifa_cop"].sum()) < 1e-6, "Debe haber UN total general (mes y barrio NULL) igual al total del hecho."
assert len(_r) == len(_det) + len(_sub) + 1, "El roll-up debe tener detalle + subtotales por mes + 1 total general."
assert list(df_slice.columns) == ["dia_semana", "viajes"], "df_slice debe tener las columnas dia_semana y viajes."
_s = _v[_v["mes"] == 3].groupby("dia_semana").size()
assert list(df_slice["dia_semana"]) == list(_s.index) and list(df_slice["viajes"]) == list(_s.values), "La rebanada de marzo debe coincidir con pandas, ordenada por dia_semana."
assert list(df_dice.columns) == ["plan", "mes", "viajes", "ingreso"], "df_dice debe tener las columnas plan, mes, viajes e ingreso."
_dd = _v[_v["plan"].isin(["anual", "estudiante"]) & _v["mes"].isin([3, 4])].groupby(["plan", "mes"])["tarifa_cop"].agg(["count", "sum"]).reset_index()
assert len(df_dice) == len(_dd) == 4 and list(df_dice["viajes"]) == list(_dd["count"]) and np.allclose(df_dice["ingreso"], _dd["sum"]), "El dado (plan ∈ {anual, estudiante}, mes ∈ {3, 4}) debe coincidir con pandas: 4 filas."
assert {"mes", "barrio_origen", "plan", "tarifa_cop"} <= set(cubo.columns) and len(cubo) == len(viajes), "cubo debe tener una fila por viaje con mes, barrio_origen, plan y tarifa_cop."
assert pivote is not None and "Total" in pivote.index and "Total" in pivote.columns, "El pivote necesita márgenes: margins=True y margins_name='Total'."
_p = _v.pivot_table(index="barrio_origen", columns="plan", values="tarifa_cop", aggfunc="sum", margins=True, margins_name="Total", fill_value=0)
assert set(pivote.index) == set(_p.index) and set(pivote.columns) == set(_p.columns) and np.allclose(pivote.sort_index().sort_index(axis=1).to_numpy(), _p.sort_index().sort_index(axis=1).to_numpy()), "El pivote debe coincidir con el calculado desde 'viajes' (suma de tarifa_cop por barrio y plan, con ceros)."
assert abs(pivote.loc["Total", "Total"] - _tot_r["ingreso"].iloc[0]) < 1e-6, "El total del pivote debe coincidir con el total general del roll-up (conciliación SQL frente a pandas)."
print(f"✅ Reto 3 superado: roll-up de {len(df_rollup)} filas (detalle + subtotales + total), total {_tot_r['ingreso'].iloc[0]:,.0f} COP | pivote {pivote.shape[0] - 1} barrios × {pivote.shape[1] - 1} planes cuadra con SQL")

---
## Reto 4: Del almacén al lago: particiones, poda y compactación 🏞️

**Contexto:** la red decide guardar sus viajes en un **lago** de archivos Parquet en vez de (solo) en el almacén. Las consultas casi siempre filtran por mes, así que el lago se **particiona estilo Hive**: `RAIZ_LAGO/anio=2025/mes=03/part-000.parquet`. Los valores de `anio` y `mes` viven en la **ruta**, no dentro de los archivos.

**Enunciado**
1. `escribir_lago(df, raiz, archivos_por_particion=3)`: para cada (`anio`, `mes`) de `df["inicio"]` escribe `archivos_por_particion` archivos `part-000.parquet`, `part-001.parquet`... (reparte las filas de la partición con `np.array_split` sobre posiciones), **sin** las columnas `anio` ni `mes`; devuelve el número de archivos. Escribe `viajes`.
2. `catalogo(raiz)` → `DataFrame` con una fila por archivo y columnas `anio` (int), `mes` (int) y `ruta` (texto), leyendo solo los nombres de las carpetas. `podar(cat, anio=None, mes=None)` → subconjunto del catálogo que cumple los filtros indicados. `leer(cat)` → `DataFrame` con el contenido de los archivos del catálogo **más** las columnas `anio` y `mes` tomadas de la ruta.
3. **Poda:** `cat_inicial`, `archivos_total`; `cat_mar = podar(cat_inicial, mes=3)`, `archivos_leidos`; `df_mar = leer(cat_mar)` y `res_poda = (filas, suma de tarifa_cop)` de marzo; `res_completo`: lo mismo leyendo **todo** el catálogo y filtrando `mes == 3` después.
4. **DuckDB:** `sql_hive` que calcule `n` e `ingreso` de marzo con `read_parquet('RAIZ_LAGO/*/*/*.parquet', hive_partitioning = true, union_by_name = true)` y `WHERE mes = 3`; `res_duck = (n, ingreso)`.
5. `compactar(raiz)`: deja **un solo archivo** `part-000.parquet` por partición (sin perder filas) y devuelve el número de archivos finales. Úsala y guarda `archivos_tras`.
6. **Esquema en lectura:** `lote_nuevo` (julio) trae una columna extra `bateria_pct`. Escríbelo con `escribir_lago` y compáctalo; `todo = leer(catalogo(RAIZ_LAGO))` y `nulos_bateria` (filas sin `bateria_pct`).

**Criterios de aceptación:** 18 archivos iniciales y 3 leídos al filtrar marzo, con el **mismo resultado** que `pandas`, que DuckDB y que leer todo y filtrar; tras compactar queda 1 archivo por partición con las mismas filas; y el lector tolera la columna nueva (los archivos viejos aportan nulos).

**Puntaje sugerido:** 20 puntos.

In [ ]:
shutil.rmtree(RAIZ_LAGO, ignore_errors=True)       # (dado) deja el lago vacío para poder re-ejecutar la celda
os.makedirs(RAIZ_LAGO)


def escribir_lago(df, raiz, archivos_por_particion=3):
    # TODO: una carpeta anio=AAAA/mes=MM por partición, con varios part-NNN.parquet (sin las columnas anio ni mes)
    return None


def catalogo(raiz):
    # TODO: DataFrame con columnas anio, mes y ruta (un archivo por fila)
    return None


def podar(cat, anio=None, mes=None):
    # TODO: filas del catálogo que cumplen los filtros dados (None = sin filtro)
    return None


def leer(cat):
    # TODO: concatena los archivos del catálogo y añade las columnas anio y mes desde la ruta
    return None


def compactar(raiz):
    # TODO: un solo part-000.parquet por partición, sin perder filas; devuelve el nº de archivos finales
    return None


# TODO: completa
n_escritos = None                      # escribir_lago(viajes, RAIZ_LAGO)
cat_inicial = archivos_total = None
cat_mar = archivos_leidos = df_mar = res_poda = None   # res_poda = (filas, suma de tarifa_cop) de marzo
res_completo = None                                    # lo mismo, leyendo los 18 archivos y filtrando mes == 3 después
sql_hive = res_duck = None                             # res_duck = (n, ingreso) de marzo con DuckDB
archivos_tras = None                                   # compactar(RAIZ_LAGO)
todo = nulos_bateria = None                            # lote_nuevo escrito + compactado; lectura de todo el lago

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert n_escritos == 18 and archivos_total == 18, "escribir_lago debe escribir 3 archivos por partición × 6 particiones (ene-jun) = 18 archivos, y el catálogo debe verlos."
assert isinstance(cat_inicial, pd.DataFrame) and list(cat_inicial.columns) == ["anio", "mes", "ruta"], "catalogo debe devolver un DataFrame con las columnas anio, mes y ruta."
assert sorted(cat_inicial["mes"].unique()) == [1, 2, 3, 4, 5, 6] and set(cat_inicial["anio"]) == {2025}, "El catálogo inicial debe cubrir los meses 1 a 6 de 2025."
assert all(f"anio={a}" in r and f"mes={m:02d}" in r for a, m, r in cat_inicial.itertuples(index=False)), "Las rutas deben seguir el estilo Hive: .../anio=2025/mes=03/part-000.parquet (mes con dos dígitos)."
assert {"anio", "mes"}.isdisjoint(pd.read_parquet(cat_inicial["ruta"].iloc[0]).columns), "Los archivos no deben guardar anio ni mes: esos valores viven en la ruta (partición)."
assert len(podar(cat_inicial)) == 18 and len(podar(cat_inicial, anio=2025, mes=3)) == 3 and len(podar(cat_inicial, anio=2024)) == 0, "podar sin filtros devuelve todo; con anio=2025, mes=3 devuelve 3 archivos; un año sin datos devuelve 0."
assert archivos_leidos == 3 and set(cat_mar["mes"]) == {3}, "Filtrar marzo debe leer solo los 3 archivos de la partición mes=3 (de 18): ese es el beneficio de la poda."
_m = viajes[viajes["inicio"].dt.month == 3]
assert res_poda is not None and res_poda[0] == len(_m) and abs(res_poda[1] - _m["tarifa_cop"].sum()) < 1e-6, "El resultado leyendo solo marzo debe coincidir con pandas sobre 'viajes'."
assert {"anio", "mes"} <= set(df_mar.columns) and (df_mar["mes"] == 3).all(), "leer debe añadir las columnas anio y mes tomadas de la ruta."
assert res_completo is not None and res_completo[0] == res_poda[0] and abs(res_completo[1] - res_poda[1]) < 1e-6, "Leer los 18 archivos y filtrar mes == 3 después debe dar lo mismo que leer solo los 3 podados (pero leyendo 6 veces más archivos)."
assert res_duck is not None and sql_hive and "hive_partitioning" in sql_hive.lower() and "union_by_name" in sql_hive.lower(), "sql_hive debe usar read_parquet con hive_partitioning = true y union_by_name = true."
assert int(res_duck[0]) == res_poda[0] and abs(float(res_duck[1]) - res_poda[1]) < 1e-6, "DuckDB, con la partición como columna, debe dar el mismo resultado que la lectura podada."
assert archivos_tras == 6, "compactar debe dejar 1 archivo por partición: 6 archivos tras compactar ene-jun."
_cat_f = catalogo(RAIZ_LAGO)
assert len(_cat_f) == 7 and all(Path(r).name == "part-000.parquet" for r in _cat_f["ruta"]), "Con julio compactado debe haber 7 particiones con un solo part-000.parquet cada una."
assert len(todo) == len(viajes) + len(lote_nuevo) == 3300, "Compactar no debe perder filas: 3 000 de ene-jun + 300 de julio."
assert "bateria_pct" in todo.columns and nulos_bateria == len(viajes) and todo["bateria_pct"].notna().sum() == len(lote_nuevo), "El lector debe tolerar la columna nueva: los 3 000 viajes viejos quedan con bateria_pct nulo y los 300 de julio con valor."
assert abs(todo["tarifa_cop"].sum() - (viajes["tarifa_cop"].sum() + lote_nuevo["tarifa_cop"].sum())) < 1e-6, "El lago compactado debe sumar lo mismo que los datos originales."
print(f"✅ Reto 4 superado: poda {archivos_leidos}/{archivos_total} archivos, marzo = {res_poda[0]} viajes ({res_poda[1]:,.0f} COP) igual en pandas y DuckDB | compactado a {archivos_tras} archivos | {nulos_bateria} nulos de bateria_pct por esquema en lectura")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué defectos corrigió tu ETL y cuáles mandó a cuarentena? ¿Por qué no se borran las filas rechazadas y qué prueba que no se perdió ninguna (usa tu `recibo`)?
- ¿Para qué sirven la dimensión «No identificado» (`sk_usuario = 0`) y las restricciones `FOREIGN KEY` y `CHECK`? ¿Qué habría pasado en el reto 2 con un `INNER JOIN` en vez de `LEFT JOIN` para el usuario?
- ¿Qué diferencia hay entre el detalle, los subtotales y el total general del *roll-up* y cómo los conciliaste con `pandas`?
- ¿Cuántos archivos evitó leer la poda de particiones y por qué solo funciona con filtros sobre la columna de partición? ¿Qué problema resuelve compactar y qué costo tiene?
- Con `bateria_pct` apareciendo en julio, ¿qué habría hecho un almacén con esquema al escribir y qué hizo el lago con esquema en lectura? ¿Cuál de los dos usarías para los reportes de la gerencia y cuál para datos nuevos y cambiantes?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: ETL que corrige lo inequívoco, rechaza con motivo y cuadra el recibo | 20 |
| Reto 2: esquema estrella en DuckDB con claves sustitutas, FK, CHECK y sin huérfanos | 25 |
| Reto 3: operaciones OLAP (roll-up, slice, dice, pivot) conciliadas entre SQL y `pandas` | 20 |
| Reto 4: lago Parquet estilo Hive, poda de particiones, compactación y esquema en lectura | 20 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] El recibo del ETL cuadra (entran = limpios + cuarentena) y no borré filas rechazadas.
- [ ] Concilié SQL con `pandas` (totales, subtotales y pivote).
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Big Data</i>
  </p>
</div>